# ITM 454 - Natural Language Processing
## Context-Free Grammars (CFGs)

---

Previous POS-tagging using NLTK

In [31]:
from nltk import word_tokenize, pos_tag

sentences = [
    "The park is beautiful",
    "I park my car here", 
    "She saw the man with binoculars",
    "I can run fast",
    "I need a can opener",
    "The book is on the table",
    "Book that flight now",
    "Time flies like an arrow",
    "Fruit flies like bananas"
]

for sentence in sentences:
    tokens = word_tokenize(sentence)
    pos_tags = pos_tag(tokens)
    print(f"Sentence: {sentence}")
    print(f"{pos_tags}")
    print()

Sentence: The park is beautiful
[('The', 'DT'), ('park', 'NN'), ('is', 'VBZ'), ('beautiful', 'JJ')]

Sentence: I park my car here
[('I', 'PRP'), ('park', 'VBP'), ('my', 'PRP$'), ('car', 'NN'), ('here', 'RB')]

Sentence: She saw the man with binoculars
[('She', 'PRP'), ('saw', 'VBD'), ('the', 'DT'), ('man', 'NN'), ('with', 'IN'), ('binoculars', 'NNS')]

Sentence: I can run fast
[('I', 'PRP'), ('can', 'MD'), ('run', 'VB'), ('fast', 'RB')]

Sentence: I need a can opener
[('I', 'PRP'), ('need', 'VBP'), ('a', 'DT'), ('can', 'MD'), ('opener', 'VB')]

Sentence: The book is on the table
[('The', 'DT'), ('book', 'NN'), ('is', 'VBZ'), ('on', 'IN'), ('the', 'DT'), ('table', 'NN')]

Sentence: Book that flight now
[('Book', 'NNP'), ('that', 'WDT'), ('flight', 'NN'), ('now', 'RB')]

Sentence: Time flies like an arrow
[('Time', 'NNP'), ('flies', 'NNS'), ('like', 'IN'), ('an', 'DT'), ('arrow', 'NN')]

Sentence: Fruit flies like bananas
[('Fruit', 'NNP'), ('flies', 'NNS'), ('like', 'IN'), ('bananas', '

### 1. Customize CFG

In [32]:
from nltk import CFG, RecursiveDescentParser

# Sentence could be NP + VP or VP + NP or VP or VP + Adv
custom_grammar = CFG.fromstring("""
    S -> NP VP | VP NP | VP | VP Adv | NP VP Adv
                                
    NP -> Det N | Det N PP | Pron | N | Det Adj N | PossDet N | Det N N
    VP -> V | V Adj | V NP | V NP PP | V PP | Aux V Adj
    PP -> P NP
                                
    Det -> "the" | "a" | "an" | "that"
    PossDet -> "my"
    N -> "park" | "car" | "man" | "binoculars" | "can" | "opener" | "book" | "table" | "flight" | "time" | "flies" | "arrow" | "fruit" | "bananas"
    Pron -> "i" | "she"
    Adj -> "beautiful" | "fast"
    V -> "is" | "park" | "saw" | "can" | "run" | "need" | "book" | "flies" | "like"
    Aux -> "can"
    P -> "with" | "on" | "like"
    Adv -> "here" | "now"
""")

# Parse the sentence
rd_parser = RecursiveDescentParser(custom_grammar)

for i in range(len(sentences)):
    tokens = sentences[i].lower().split()
    print(f"Sentence {i+1}: {sentences[i]}")
    for tree in rd_parser.parse(tokens):
        print(tree)
    print("-----------")

# How about sentences 2 and 5? Which rules are we missing?
'''
For sentence 2: 
 "I park my car here"
- The missing rule:  "NP VP Adv" to S
- Tokens => i(Pron) park(V) my(PossDet) car(N) here(Adv)
- Problem: The 2nd sentence grammar had n S rule that have a leading NP, a VP, and a trailing Adv. because it requires the sentences to start with 
a verb, not just "I".
    - NP  -> Pron -> "i"
    - VP  -> V NP -> "park" + NP("my car"), where NP -> PossDet N -> "my" "car"
    - Adv -> "here"
- So S -> NP VP Adv now derives "i" + "park my car" + "here".
'''

'''
For sentence 5: 
 "I need a can opener"
- The missing rule:  "Det N N" to NP
- Tokens => i(Pron) need(V) a(Det) can(N) opener(N)
- Problem: The NP rule of the 5th sentence only allowed a single noun after a determiner, which it couldnt parse the 
noun-noun like "can opener".
    - NP -> Pron -> "i"
    - VP -> V NP -> "need" + NP("a can opener"),
         where NP -> Det N N -> "a" + "can" + "opener"
- So S -> NP VP now derives "i" + "need a can opener".
'''


Sentence 1: The park is beautiful
(S (NP (Det the) (N park)) (VP (V is) (Adj beautiful)))
-----------
Sentence 2: I park my car here
(S (NP (Pron i)) (VP (V park) (NP (PossDet my) (N car))) (Adv here))
-----------
Sentence 3: She saw the man with binoculars
(S
  (NP (Pron she))
  (VP
    (V saw)
    (NP (Det the) (N man) (PP (P with) (NP (N binoculars))))))
(S
  (NP (Pron she))
  (VP
    (V saw)
    (NP (Det the) (N man))
    (PP (P with) (NP (N binoculars)))))
-----------
Sentence 4: I can run fast
(S (NP (Pron i)) (VP (Aux can) (V run) (Adj fast)))
-----------
Sentence 5: I need a can opener
(S (NP (Pron i)) (VP (V need) (NP (Det a) (N can) (N opener))))
-----------
Sentence 6: The book is on the table
(S
  (NP (Det the) (N book))
  (VP (V is) (PP (P on) (NP (Det the) (N table)))))
-----------
Sentence 7: Book that flight now
(S (VP (V book) (NP (Det that) (N flight))) (Adv now))
-----------
Sentence 8: Time flies like an arrow
(S
  (NP (N time))
  (VP (V flies) (PP (P like) (NP (Det

'\nFor sentence 5: \n "I need a can opener"\n- The missing rule:  "Det N N" to NP\n- Tokens => i(Pron) need(V) a(Det) can(N) opener(N)\n- Problem: The NP rule of the 5th sentence only allowed a single noun after a determiner, which it couldnt parse the \nnoun-noun like "can opener".\n    - NP -> Pron -> "i"\n    - VP -> V NP -> "need" + NP("a can opener"),\n         where NP -> Det N N -> "a" + "can" + "opener"\n- So S -> NP VP now derives "i" + "need a can opener".\n'

### 2. Treebank CFG

In [33]:
import nltk
nltk.download('treebank')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package treebank to
[nltk_data]     /Users/soryapheakly/nltk_data...
[nltk_data]   Package treebank is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/soryapheakly/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/soryapheakly/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/soryapheakly/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/soryapheakly/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [34]:
from nltk import nonterminals, induce_pcfg, ViterbiParser, pos_tag
from nltk.corpus import treebank

S, NP, VP, PP = nonterminals('S, NP, VP, PP')
N, V, P, DT = nonterminals('N, V, P, DT')

productions = []
# select only the first 5 files from the treebank for faster processing
for fileid in treebank.fileids()[:5]:
    for t in treebank.parsed_sents(fileid):
        productions += t.productions()
grammar = induce_pcfg(S, productions)

# Check vocabulary - to ensure the words are in there
vocabulary = set()
for prod in grammar.productions():
    if prod.is_lexical():
        vocabulary.update(prod.rhs())
print(vocabulary)

viterbi_parser = ViterbiParser(grammar)

sent1 = "The research continued"
print(f'{sent1}:')
for tree in viterbi_parser.parse(sent1.split()):
    print(tree)
print("-----------")

sent2 = "The National University support research overseas"
print(f'{sent2}:')
for tree in viterbi_parser.parse(sent2.split()):
    print(tree)
print("-----------")

{'seven', 'classified', 'questionable', '8.22', 'Although', 'force', 'he', 'who', 'nations', 'explained', '8.45', 'are', '*T*-2', 'asbestos', 'down', '9', 'recognize', 'IBC', 'show', 'president', 'brief', 'most', 'waiving', 'said', 'worked', 'British', '*T*-9', 'has', 'had', 'It', 'Medicine', '29', 'a', 'as', 'area', 'bin', 'making', 'found', 'Brenda', 'Journal', '*?*', 'bearing', 'fans', 'longer', 'market', '*T*-8', 'because', 'stringently', 'spokeswoman', 'which', 'under', 'D.', 'funds', "'", 'team', '61', 'than', 'pour', 'mutual', '8.14', 'story', 'whether', 'Dreyfus', 'Gold', 'aware', 'sold', '1997', 'with', '8.04', 'makes', 'acetate', 'England', 'malignant', 'though', 'workers', 'factory', 'anyone', 'conglomerate', 'events', 'its', 'James', 'Nov.', 'thought', 'during', 'now', ',', 'high', 'Lorillard', 'dumped', 'probably', 'higher', 'average', 'elected', 'N.V.', 'study', 'Longer', 'properties', 'Talcott', '*T*-10', 'was', 'gradual', 'tracked', 'mixed', 'industrial', 'currently', '

### 3. Dependency Parsing 

#### 1. Install the spacy

In [35]:
%pip install spacy
!python -m spacy download en_core_web_sm

Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 355.8 kB/s  0:00:330:00:0100:02
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [36]:
import spacy
from spacy.tokens import Doc
from spacy.matcher import Matcher
from spacy import displacy

#### 2. Load the spaCy language model

In [37]:
nlp = spacy.load("en_core_web_sm")

#### 3. Define the info extraction func

In [ ]:
def information_extraction(doc):
    matcher = Matcher(nlp.vocab)

    svo_pattern = [
        {"DEP": "nsubj"},
        {"DEP": "aux", "OP": "?"},
        {"POS": "VERB"},
        {"DEP": "det", "OP": "?"},
        {"DEP": {"IN": ["dobj", "attr", "pobj"]}}
    ]

#### 4. Create the SVO pattern

In [39]:
svo_pattern = [
        {"DEP": "nsubj"},
        {"DEP": "aux", "OP": "?"},
        {"POS": "VERB"},
        {"DEP": "det", "OP": "?"},
        {"DEP": {"IN": ["dobj", "attr", "pobj"]}}
    ]

#### 5. Add the pattern and find the match

In [40]:
matcher = Matcher(nlp.vocab)
doc = nlp("I need a can opener")  # Using your sentence 5 as a test case

#### 6. Extract Sub, V and obj

In [ ]:
relations = []
for _, start, end in matches:
    span = doc[start:end]
    subj = span[0].text
    verb = span[2].lemma_
    obj = span[-1].text
    relations.append((subj, verb, obj))

return relations

IndentationError: unexpected indent (4137385368.py, line 2)

#### 7. Register Custom Doc extension

#### 8. Provide input text and process it

#### 9. Print Named Entities

#### 10. Visualize the dependency and entities